# Notebook 12 — Build H₂ Pipeline Capacity-Cost Dataset

This notebook converts the normalized hydrogen pipeline cost worksheets from the master Excel cost model into a single standardized and traceable capacity-cost dataset.

## Purpose

The notebook prepares the engineering cost observations needed by the downstream cost-function and schema-building workflow. It does not fit regression functions or write directly to the CANOE/TEMOA SQLite schema.

## Inputs

The master H₂ pipeline cost workbook contains separate normalized worksheets for:

- pipeline capital expenditure;
- fixed operating expenditure;
- variable operating expenditure.

Each worksheet reports costs against a common set of pipeline inner diameters and annual hydrogen transport capacities.

## Processing steps

1. Locate and validate the master cost workbook.
2. Load the normalized CAPEX, fixed OPEX, and variable OPEX worksheets.
3. Remove completely blank rows and columns.
4. Standardize worksheet column names and cost units.
5. Confirm that diameter and annual-capacity cases match across all worksheets.
6. Merge the three cost datasets into one canonical capacity-cost table.
7. Add technology, commodity, currency, source, and cost-model metadata.
8. Export the processed table as a CSV intermediate product.

## Output

The exported table contains one row per pipeline diameter and capacity case, with:

- `technology`
- `commodity`
- `diameter_in`
- `capacity_t_h2_per_year`
- `total_capex_cad2020_per_km`
- `total_variable_opex_cad2020_per_year_per_km`
- `total_fixed_opex_cad2020_per_year_per_km`
- currency and source-model metadata

This processed dataset becomes the input to Notebook 13, which fits candidate cost functions and constructs schema-facing pipeline cost tables.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

In [2]:
def find_project_root(start_path: Path | None = None) -> Path:
    """Find the Geospatial-CANOE repository root.

    Searches upward from the supplied path, or from the current working
    directory when no path is supplied. The first directory containing both
    ``scripts`` and ``data_files`` is treated as the project root.

    Parameters
    ----------
    start_path : Path | None, default None
        Directory from which to begin searching.

    Returns
    -------
    Path
        Absolute path to the repository root.

    Raises
    ------
    FileNotFoundError
        If the repository root cannot be located.
    """

    start = Path.cwd().resolve() if start_path is None else start_path.resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "scripts").exists()
            and (candidate / "data_files").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate project root. "
        "Expected to find scripts/ and data_files/."
    )

In [3]:
# Check that the project root can be found when this script is run directly.

PROJECT_ROOT = find_project_root()

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace


In [4]:
# =============================================================================
# Input and output paths
# =============================================================================

MASTER_COST_MODEL_DIR = (
    PROJECT_ROOT
    / "data_files"
    / "models"
    / "cost_models"
    / "transport"
    / "master_files"
)

PROCESSED_COST_DIR = (
    PROJECT_ROOT
    / "data_files"
    / "processed"
    / "costs"
    / "transport"
    / "pipelines"
    / "h2_pipeline"
)

WORKBOOK_PATH = (
    MASTER_COST_MODEL_DIR
    / "h2_pipeline_costs_master_v2.xlsx"
)

OUTPUT_CSV_PATH = (
    PROCESSED_COST_DIR
    / "h2_pipeline_normalized_capacity_costs.csv"
)

In [5]:
# Check that the master cost workbook exists and create the output directory if it does not exist.

if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(
        f"Master cost workbook not found: {WORKBOOK_PATH}"
    )

PROCESSED_COST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Workbook: {WORKBOOK_PATH}")
print(f"Output directory: {PROCESSED_COST_DIR}")
print(f"Output CSV: {OUTPUT_CSV_PATH}")

Workbook: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\models\cost_models\transport\master_files\h2_pipeline_costs_master_v2.xlsx
Output directory: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline
Output CSV: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_normalized_capacity_costs.csv


In [6]:
# =============================================================================
# Workbook worksheet inspection
# =============================================================================

def list_workbook_sheets(workbook_path: Path) -> list[str]:
    """Return worksheet names from an Excel workbook.

    Parameters
    ----------
    workbook_path : Path
        Path to the source Excel workbook.

    Returns
    -------
    list[str]
        Worksheet names in workbook order.

    Raises
    ------
    FileNotFoundError
        If the workbook does not exist.
    """

    if not workbook_path.exists():
        raise FileNotFoundError(
            f"Workbook not found: {workbook_path}"
        )

    excel_file = pd.ExcelFile(workbook_path)

    return excel_file.sheet_names


sheet_names = list_workbook_sheets(WORKBOOK_PATH)

print("Workbook worksheets:")

for index, sheet_name in enumerate(sheet_names):
    print(f"  [{index}] {sheet_name}")


NORMALIZED_COST_SHEETS = {
    "capex": "Normalized CAPEX",
    "variable_opex": "Normalized Variable OPEX",
    "fixed_opex": "Normalized Fixed OPEX",
}

missing_sheets = [
    sheet_name
    for sheet_name in NORMALIZED_COST_SHEETS.values()
    if sheet_name not in sheet_names
]

if missing_sheets:
    raise ValueError(
        "Missing required normalized cost worksheet(s): "
        f"{missing_sheets}. "
        f"Available worksheets: {sheet_names}"
    )

print("\nSelected normalized cost worksheets:")

for cost_type, sheet_name in NORMALIZED_COST_SHEETS.items():
    print(f"  {cost_type}: {sheet_name}")

Workbook worksheets:
  [0] README
  [1] Source_Index
  [2] Assumptions and Parameters Pipe
  [3] Transition Accelerator Pipe
  [4] Assumptions and Parameter Comp
  [5] Transition Accelerator Compress
  [6] Normalized Transmission Costs
  [7] Normalized CAPEX
  [8] Normalized Variable OPEX
  [9] Normalized Fixed OPEX

Selected normalized cost worksheets:
  capex: Normalized CAPEX
  variable_opex: Normalized Variable OPEX
  fixed_opex: Normalized Fixed OPEX


In [7]:
# =============================================================================
# Load normalized pipeline cost worksheets
# =============================================================================

def load_normalized_cost_sheets(
    workbook_path: Path,
    sheet_map: dict[str, str],
) -> dict[str, pd.DataFrame]:
    """Load and lightly clean normalized pipeline cost worksheets.

    Parameters
    ----------
    workbook_path : Path
        Path to the master pipeline cost workbook.
    sheet_map : dict[str, str]
        Mapping from canonical cost-type names to Excel worksheet names.

    Returns
    -------
    dict[str, pd.DataFrame]
        Normalized cost tables keyed by canonical cost type.

    Raises
    ------
    FileNotFoundError
        If the workbook does not exist.
    ValueError
        If no worksheets are requested, a requested worksheet is missing,
        or a loaded worksheet contains no data.
    """

    if not workbook_path.exists():
        raise FileNotFoundError(
            f"Pipeline cost workbook not found: {workbook_path}"
        )

    if not sheet_map:
        raise ValueError("No normalized cost worksheets were specified.")

    cost_tables: dict[str, pd.DataFrame] = {}

    with pd.ExcelFile(workbook_path) as workbook:
        available_sheets = set(workbook.sheet_names)
        requested_sheets = set(sheet_map.values())

        missing_sheets = sorted(requested_sheets - available_sheets)

        if missing_sheets:
            raise ValueError(
                "The pipeline cost workbook is missing required worksheet(s): "
                f"{missing_sheets}"
            )

        for cost_type, sheet_name in sheet_map.items():
            cost_table = pd.read_excel(
                workbook,
                sheet_name=sheet_name,
            )

            cost_table = (
                cost_table
                .dropna(axis="index", how="all")
                .dropna(axis="columns", how="all")
                .reset_index(drop=True)
            )

            if cost_table.empty:
                raise ValueError(
                    f"Worksheet '{sheet_name}' contains no usable data."
                )

            cost_tables[cost_type] = cost_table

    return cost_tables


normalized_cost_tables = load_normalized_cost_sheets(
    workbook_path=WORKBOOK_PATH,
    sheet_map=NORMALIZED_COST_SHEETS,
)

print("Loaded normalized cost tables:")

for cost_type, cost_table in normalized_cost_tables.items():
    sheet_name = NORMALIZED_COST_SHEETS[cost_type]

    print(
        f"  {cost_type}: "
        f"'{sheet_name}' — "
        f"{cost_table.shape[0]:,} rows × "
        f"{cost_table.shape[1]:,} columns"
    )


Loaded normalized cost tables:
  capex: 'Normalized CAPEX' — 13 rows × 3 columns
  variable_opex: 'Normalized Variable OPEX' — 13 rows × 3 columns
  fixed_opex: 'Normalized Fixed OPEX' — 13 rows × 3 columns


In [8]:
# =============================================================================
# Inspect loaded normalized pipeline cost worksheets
# =============================================================================

for cost_type, cost_table in normalized_cost_tables.items():
    print("\n" + "=" * 80)
    print(f"{cost_type}: {NORMALIZED_COST_SHEETS[cost_type]}")
    print("=" * 80)

    display(cost_table.head(10))
    print("\nColumns:")
    print(cost_table.columns.tolist())


capex: Normalized CAPEX


,"Pipeline sizes (inch, inner diameter)",Annual Capacity (t H2 / year),Total CAPEX ($ 2020 CAD / km)
0,8,5.397953e+04,1.394121e+06
1,10,9.429835e+04,1.632461e+06
2,12,1.487501e+05,1.880928e+06
3,14,2.186876e+05,2.140113e+06
4,16,3.053543e+05,2.410694e+06
5,18,4.099070e+05,2.694416e+06
6,20,5.334320e+05,2.991624e+06
7,22,6.769566e+05,3.303302e+06
8,24,8.414575e+05,3.630651e+06
9,26,1.027868e+06,3.975046e+06



Columns:
['Pipeline sizes (inch, inner diameter)', 'Annual Capacity (t H2 / year)', 'Total CAPEX ($ 2020 CAD / km)']

variable_opex: Normalized Variable OPEX


,"Pipeline sizes (inch, inner diameter)",Annual Capacity (t H2 / year),Total Variable OPEX ($ 2020 CAD/ km)
0,8,5.397953e+04,10333.906985
1,10,9.429835e+04,17901.573988
2,12,1.487501e+05,27961.113404
3,14,2.186876e+05,40711.907844
4,16,3.053543e+05,56335.036261
5,18,4.099070e+05,75167.744427
6,20,5.334320e+05,97417.863378
7,22,6.769566e+05,123270.426166
8,24,8.414575e+05,152901.400879
9,26,1.027868e+06,186478.906891



Columns:
['Pipeline sizes (inch, inner diameter)', 'Annual Capacity (t H2 / year)', 'Total Variable OPEX ($ 2020 CAD/ km)']

fixed_opex: Normalized Fixed OPEX


,"Pipeline sizes (inch, inner diameter)",Annual Capacity (t H2 / year),Total Fixed OPEX ($ 2020 CAD per year / km* per year capacity)
0,8,5.397953e+04,40138.383237
1,10,9.429835e+04,60354.158581
2,12,1.487501e+05,80401.712562
3,14,2.186876e+05,100426.340615
4,16,3.053543e+05,120535.913145
5,18,4.099070e+05,140829.069306
6,20,5.334320e+05,161376.679926
7,22,6.769566e+05,182237.909868
8,24,8.414575e+05,203465.076877
9,26,1.027868e+06,225106.259138



Columns:
['Pipeline sizes (inch, inner diameter)', 'Annual Capacity (t H2 / year)', 'Total Fixed OPEX ($ 2020 CAD per year / km* per year capacity)']


In [9]:
# =============================================================================
# Standardize normalized pipeline cost-table columns
# =============================================================================

COMMON_COLUMN_MAP = {
    "Pipeline sizes (inch, inner diameter)": "diameter_in",
    "Annual Capacity (t H2 / year)": "capacity_t_h2_per_year",
}

COST_COLUMN_MAPS = {
    "capex": {
        "Total CAPEX ($ 2020 CAD / km)": (
            "total_capex_cad2020_per_km"
        ),
    },
    "variable_opex": {
        "Total Variable OPEX ($ 2020 CAD/ km)": (
            "total_variable_opex_cad2020_per_year_per_km"
        ),
    },
    "fixed_opex": {
        "Total Fixed OPEX ($ 2020 CAD per year / km* per year capacity)": (
            "total_fixed_opex_cad2020_per_year_per_km"
        ),
    },
}


def standardize_cost_table_columns(
    cost_tables: dict[str, pd.DataFrame],
    common_column_map: dict[str, str],
    cost_column_maps: dict[str, dict[str, str]],
) -> dict[str, pd.DataFrame]:
    """Standardize normalized pipeline cost-table column names.

    Parameters
    ----------
    cost_tables : dict[str, pd.DataFrame]
        Normalized cost tables keyed by cost type.
    common_column_map : dict[str, str]
        Column-name mapping shared by all cost tables.
    cost_column_maps : dict[str, dict[str, str]]
        Cost-specific column-name mappings keyed by cost type.

    Returns
    -------
    dict[str, pd.DataFrame]
        Cost tables containing only required columns with standardized names.

    Raises
    ------
    ValueError
        If a cost type has no column mapping, a required column is missing,
        or standardized column names are duplicated.
    """

    standardized_tables: dict[str, pd.DataFrame] = {}

    for cost_type, cost_table in cost_tables.items():
        if cost_type not in cost_column_maps:
            raise ValueError(
                f"No cost-column mapping defined for '{cost_type}'."
            )

        column_map = {
            **common_column_map,
            **cost_column_maps[cost_type],
        }

        missing_columns = [
            source_column
            for source_column in column_map
            if source_column not in cost_table.columns
        ]

        if missing_columns:
            raise ValueError(
                f"{cost_type} table is missing required columns: "
                f"{missing_columns}"
            )

        standardized_columns = list(column_map.values())

        if len(standardized_columns) != len(set(standardized_columns)):
            raise ValueError(
                f"{cost_type} column mapping produces duplicate "
                "standardized column names."
            )

        standardized_table = (
            cost_table
            .loc[:, list(column_map)]
            .rename(columns=column_map)
            .copy()
        )

        standardized_tables[cost_type] = standardized_table

    return standardized_tables


standardized_cost_tables = standardize_cost_table_columns(
    cost_tables=normalized_cost_tables,
    common_column_map=COMMON_COLUMN_MAP,
    cost_column_maps=COST_COLUMN_MAPS,
)

print("Standardized cost-table columns:")

for cost_type, cost_table in standardized_cost_tables.items():
    print(
        f"  {cost_type}: "
        f"{cost_table.columns.tolist()}"
    )

Standardized cost-table columns:
  capex: ['diameter_in', 'capacity_t_h2_per_year', 'total_capex_cad2020_per_km']
  variable_opex: ['diameter_in', 'capacity_t_h2_per_year', 'total_variable_opex_cad2020_per_year_per_km']
  fixed_opex: ['diameter_in', 'capacity_t_h2_per_year', 'total_fixed_opex_cad2020_per_year_per_km']


In [10]:
# =============================================================================
# Validate shared pipeline engineering cases
# =============================================================================

KEY_COLUMNS = [
    "diameter_in",
    "capacity_t_h2_per_year",
]


def validate_matching_cost_table_keys(
    cost_tables: dict[str, pd.DataFrame],
    reference_cost_type: str,
    key_columns: list[str],
) -> None:
    """Validate that all cost tables contain the same engineering cases.

    Parameters
    ----------
    cost_tables : dict[str, pd.DataFrame]
        Standardized cost tables keyed by cost type.
    reference_cost_type : str
        Cost table used as the reference case set.
    key_columns : list[str]
        Shared engineering columns used to identify pipeline cases.

    Raises
    ------
    ValueError
        If the reference table is missing, a comparison table has duplicate
        keys, or its engineering cases differ from the reference table.
    """

    if reference_cost_type not in cost_tables:
        raise ValueError(
            f"Reference cost table '{reference_cost_type}' was not found."
        )

    reference_keys = (
        cost_tables[reference_cost_type][key_columns]
        .drop_duplicates()
        .sort_values(key_columns)
        .reset_index(drop=True)
    )

    if len(reference_keys) != len(cost_tables[reference_cost_type]):
        raise ValueError(
            f"Reference table '{reference_cost_type}' contains duplicate "
            f"engineering cases based on {key_columns}."
        )

    for cost_type, cost_table in cost_tables.items():
        if cost_type == reference_cost_type:
            continue

        comparison_keys = (
            cost_table[key_columns]
            .drop_duplicates()
            .sort_values(key_columns)
            .reset_index(drop=True)
        )

        if len(comparison_keys) != len(cost_table):
            raise ValueError(
                f"Table '{cost_type}' contains duplicate engineering cases "
                f"based on {key_columns}."
            )

        if not reference_keys.equals(comparison_keys):
            missing_from_comparison = (
                reference_keys
                .merge(
                    comparison_keys,
                    on=key_columns,
                    how="left",
                    indicator=True,
                )
                .query("_merge == 'left_only'")
                .drop(columns="_merge")
            )

            additional_in_comparison = (
                comparison_keys
                .merge(
                    reference_keys,
                    on=key_columns,
                    how="left",
                    indicator=True,
                )
                .query("_merge == 'left_only'")
                .drop(columns="_merge")
            )

            raise ValueError(
                f"Engineering cases in '{cost_type}' do not match "
                f"'{reference_cost_type}'.\n"
                f"Missing from '{cost_type}':\n"
                f"{missing_from_comparison.to_string(index=False)}\n"
                f"Additional in '{cost_type}':\n"
                f"{additional_in_comparison.to_string(index=False)}"
            )


validate_matching_cost_table_keys(
    cost_tables=standardized_cost_tables,
    reference_cost_type="capex",
    key_columns=KEY_COLUMNS,
)

print("\nDiameter and capacity values match across all cost tables.")


Diameter and capacity values match across all cost tables.


In [11]:
# =============================================================================
# Merge normalized pipeline cost tables
# =============================================================================

def merge_normalized_cost_tables(
    cost_tables: dict[str, pd.DataFrame],
    key_columns: list[str],
) -> pd.DataFrame:
    """Merge normalized pipeline cost tables into one canonical table.

    Parameters
    ----------
    cost_tables : dict[str, pd.DataFrame]
        Standardized cost tables keyed by cost type.
    key_columns : list[str]
        Shared engineering columns used to align cost cases.

    Returns
    -------
    pd.DataFrame
        Combined pipeline capacity-cost table.

    Raises
    ------
    ValueError
        If a required cost table is missing or a merge is not one-to-one.
    """

    required_cost_types = [
        "capex",
        "variable_opex",
        "fixed_opex",
    ]

    missing_cost_types = [
        cost_type
        for cost_type in required_cost_types
        if cost_type not in cost_tables
    ]

    if missing_cost_types:
        raise ValueError(
            f"Missing required cost tables: {missing_cost_types}"
        )

    combined_costs = cost_tables["capex"].copy()

    for cost_type in ["variable_opex", "fixed_opex"]:
        cost_columns = [
            column
            for column in cost_tables[cost_type].columns
            if column not in key_columns
        ]

        combined_costs = combined_costs.merge(
            cost_tables[cost_type][key_columns + cost_columns],
            on=key_columns,
            how="inner",
            validate="one_to_one",
        )

    combined_costs = (
        combined_costs
        .sort_values("capacity_t_h2_per_year")
        .reset_index(drop=True)
    )

    return combined_costs


combined_costs = merge_normalized_cost_tables(
    cost_tables=standardized_cost_tables,
    key_columns=KEY_COLUMNS,
)

print(
    "Combined normalized pipeline costs: "
    f"{combined_costs.shape[0]:,} rows × "
    f"{combined_costs.shape[1]:,} columns"
)

display(combined_costs)

Combined normalized pipeline costs: 13 rows × 5 columns


,diameter_in,capacity_t_h2_per_year,total_capex_cad2020_per_km,total_variable_opex_cad2020_per_year_per_km,total_fixed_opex_cad2020_per_year_per_km
0,8,5.397953e+04,1.394121e+06,10333.906985,40138.383237
1,10,9.429835e+04,1.632461e+06,17901.573988,60354.158581
2,12,1.487501e+05,1.880928e+06,27961.113404,80401.712562
3,14,2.186876e+05,2.140113e+06,40711.907844,100426.340615
4,16,3.053543e+05,2.410694e+06,56335.036261,120535.913145
5,18,4.099070e+05,2.694416e+06,75167.744427,140829.069306
6,20,5.334320e+05,2.991624e+06,97417.863378,161376.679926
7,22,6.769566e+05,3.303302e+06,123270.426166,182237.909868
8,24,8.414575e+05,3.630651e+06,152901.400879,203465.076877
9,26,1.027868e+06,3.975046e+06,186478.906891,225106.259138


In [12]:
# =============================================================================
# Add pipeline cost-model metadata
# =============================================================================

def add_cost_model_metadata(
    cost_table: pd.DataFrame,
    technology: str,
    commodity: str,
    currency: str,
    currency_year: int,
    source_workbook: str,
    cost_model_version: str,
) -> pd.DataFrame:
    """Add traceable metadata to a pipeline capacity-cost table.

    Parameters
    ----------
    cost_table : pd.DataFrame
        Combined pipeline capacity-cost table.
    technology : str
        CANOE/TEMOA technology identifier.
    commodity : str
        CANOE/TEMOA transported commodity identifier.
    currency : str
        Currency used for all monetary values.
    currency_year : int
        Base year used for monetary values.
    source_workbook : str
        Filename of the source master cost workbook.
    cost_model_version : str
        Version identifier for the processed cost model.

    Returns
    -------
    pd.DataFrame
        Pipeline cost table with model metadata columns added.
    """

    output_table = cost_table.copy()

    output_table.insert(0, "technology", technology)
    output_table.insert(1, "commodity", commodity)

    output_table["currency"] = currency
    output_table["currency_year"] = currency_year
    output_table["source_workbook"] = source_workbook
    output_table["cost_model_version"] = cost_model_version

    return output_table


pipeline_cost_table = add_cost_model_metadata(
    cost_table=combined_costs,
    technology="H2_PIPE",
    commodity="h2",
    currency="CAD",
    currency_year=2020,
    source_workbook=WORKBOOK_PATH.name,
    cost_model_version="v1",
)

display(pipeline_cost_table)

,technology,commodity,diameter_in,capacity_t_h2_per_year,total_capex_cad2020_per_km,total_variable_opex_cad2020_per_year_per_km,total_fixed_opex_cad2020_per_year_per_km,currency,currency_year,source_workbook,cost_model_version
0,H2_PIPE,h2,8,5.397953e+04,1.394121e+06,10333.906985,40138.383237,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
1,H2_PIPE,h2,10,9.429835e+04,1.632461e+06,17901.573988,60354.158581,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
2,H2_PIPE,h2,12,1.487501e+05,1.880928e+06,27961.113404,80401.712562,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
3,H2_PIPE,h2,14,2.186876e+05,2.140113e+06,40711.907844,100426.340615,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
4,H2_PIPE,h2,16,3.053543e+05,2.410694e+06,56335.036261,120535.913145,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
5,H2_PIPE,h2,18,4.099070e+05,2.694416e+06,75167.744427,140829.069306,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
6,H2_PIPE,h2,20,5.334320e+05,2.991624e+06,97417.863378,161376.679926,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
7,H2_PIPE,h2,22,6.769566e+05,3.303302e+06,123270.426166,182237.909868,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
8,H2_PIPE,h2,24,8.414575e+05,3.630651e+06,152901.400879,203465.076877,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
9,H2_PIPE,h2,26,1.027868e+06,3.975046e+06,186478.906891,225106.259138,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1


In [13]:
# =============================================================================
# Export pipeline capacity-cost table
# =============================================================================

def export_cost_table(
    cost_table: pd.DataFrame,
    output_path: Path,
) -> Path:
    """Export a processed pipeline capacity-cost table to CSV.

    Parameters
    ----------
    cost_table : pd.DataFrame
        Processed pipeline capacity-cost table.
    output_path : Path
        Destination CSV path.

    Returns
    -------
    Path
        Path to the exported CSV.

    Raises
    ------
    ValueError
        If the cost table is empty or contains duplicate column names.
    OSError
        If the CSV cannot be written successfully.
    """

    if cost_table.empty:
        raise ValueError("Cannot export an empty pipeline cost table.")

    duplicated_columns = (
        cost_table.columns[
            cost_table.columns.duplicated()
        ]
        .tolist()
    )

    if duplicated_columns:
        raise ValueError(
            "Pipeline cost table contains duplicate column names: "
            f"{duplicated_columns}"
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    cost_table.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    if not output_path.exists():
        raise OSError(
            f"Pipeline cost table was not created: {output_path}"
        )

    return output_path


exported_cost_path = export_cost_table(
    cost_table=pipeline_cost_table,
    output_path=OUTPUT_CSV_PATH,
)

print("Exported pipeline capacity-cost table:")
print(f"  Path: {exported_cost_path}")
print(f"  Rows: {len(pipeline_cost_table):,}")
print(f"  Columns: {len(pipeline_cost_table.columns):,}")
print(f"  Size: {exported_cost_path.stat().st_size:,} bytes")

Exported pipeline capacity-cost table:
  Path: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_normalized_capacity_costs.csv
  Rows: 13
  Columns: 11
  Size: 1,964 bytes
